In [ ]:
# Agentic AI-Based Phishing Email Detection and Explanation System
"""
This notebook implements the agentic investigation layer.

Architecture:

Email
  ↓
Parse + IOC extraction
  ↓
ML phishing detector
  ↓
Header analysis
  ↓
Jev decision model (optional)
  ↓
LangGraph investigation agent
  ↓
Tavily web search + AlienVault OTX
  ↓
Evidence collection
  ↓
Gemini investigation score
  ↓
Evidence graph
  ↓
Score fusion
  ↓
Final phishing report"""


'\nThis notebook implements the agentic investigation layer.\n\nArchitecture:\n\nEmail\n  ↓\nParse + IOC extraction\n  ↓\nML phishing detector\n  ↓\nHeader analysis\n  ↓\nJev decision model (optional)\n  ↓\nLangGraph investigation agent\n  ↓\nTavily web search + AlienVault OTX\n  ↓\nEvidence collection\n  ↓\nGemini investigation score\n  ↓\nEvidence graph\n  ↓\nScore fusion\n  ↓\nFinal phishing report'

In [ ]:
import os
import re
import json
import math
import statistics
from pathlib import Path
from typing import Any, Optional, TypedDict, Annotated

import requests
import joblib
import numpy as np
import pandas as pd
import networkx as nx

from urllib.parse import urlparse, unquote
from email import policy
from email.parser import Parser
from email.utils import parseaddr

from dotenv import load_dotenv

from IPython.display import display, HTML

from pydantic import BaseModel, Field

from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    AIMessage,
    ToolMessage,
)
from langchain_core.tools import tool

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


In [ ]:
DOTENV_PATH = Path.cwd() / ".env"

if DOTENV_PATH.exists():
    load_dotenv(DOTENV_PATH)
else:
    load_dotenv()

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")
OTX_API_KEY = os.environ.get("OTX_API_KEY")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

required = {
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "TAVILY_API_KEY": TAVILY_API_KEY,
    "OTX_API_KEY": OTX_API_KEY,
}

missing = [
    name
    for name, value in required.items()
    if not value
]

if missing:
    raise ValueError(
        f"Missing in .env: {', '.join(missing)}"
    )

USE_JEV = True

print("Gemini:", bool(GEMINI_API_KEY))
print("Tavily:", bool(TAVILY_API_KEY))
print("OTX:", bool(OTX_API_KEY))
print("Jev:", bool(USE_JEV and OPENROUTER_API_KEY))


In [ ]:
MODEL_PATH = Path.cwd() / "phishing_detector.joblib"

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Expected {MODEL_PATH}"
    )

phishing_model = joblib.load(MODEL_PATH)

print("Model loaded:")
print(type(phishing_model))


In [ ]:
TEST_EMAIL = """
From: security@example.com
Subject: Urgent account verification

Your account requires immediate verification.
Please click the following link within 24 hours:

https://example.com/verify
"""

ml_test_score = phishing_model.predict_proba([TEST_EMAIL])[0][1]

print(f"ML phishing probability: {ml_test_score:.4f}")

ML phishing probability: 0.9880


In [ ]:
def extract_body(msg):
    if msg.is_multipart():
        text_parts = []

        for part in msg.walk():
            content_type = part.get_content_type()
            disposition = str(part.get("Content-Disposition", ""))

            if (
                content_type == "text/plain"
                and "attachment" not in disposition.lower()
            ):
                try:
                    text_parts.append(part.get_content())
                except Exception:
                    payload = part.get_payload(decode=True)
                    if payload:
                        text_parts.append(
                            payload.decode(
                                errors="ignore"
                            )
                        )

        return "\n".join(text_parts).strip()

    try:
        return msg.get_content().strip()
    except Exception:
        payload = msg.get_payload(decode=True)

        if isinstance(payload, bytes):
            return payload.decode(errors="ignore")

        return str(payload or "").strip()


def parse_email(raw_email: str) -> dict:

    msg = Parser(policy=policy.default).parsestr(raw_email.lstrip())

    sender_name, sender_address = parseaddr(
        msg.get("From", "")
    )

    reply_name, reply_address = parseaddr(
        msg.get("Reply-To", "")
    )

    return {
        "subject": msg.get("Subject", ""),
        "from": msg.get("From", ""),
        "sender_name": sender_name,
        "sender_address": sender_address,
        "reply_to": msg.get("Reply-To", ""),
        "reply_to_address": reply_address,
        "return_path": msg.get("Return-Path", ""),
        "date": msg.get("Date", ""),
        "message_id": msg.get("Message-ID", ""),
        "authentication_results": msg.get(
            "Authentication-Results", ""
        ),
        "received_spf": msg.get("Received-SPF", ""),
        "body": extract_body(msg),
        "headers": dict(msg.items()),
    }

In [ ]:
URL_RE = re.compile(
    r'https?://[^\s<>"\'\]\)]+',
    re.IGNORECASE
)

IP_RE = re.compile(
    r'\b(?:\d{1,3}\.){3}\d{1,3}\b'
)

DOMAIN_RE = re.compile(
    r'\b(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}\b'
)


def clean_url(url: str) -> str:
    return url.rstrip(".,;:!?)]}")


def extract_iocs(text: str) -> dict:

    urls = sorted({
        clean_url(u)
        for u in URL_RE.findall(text or "")
    })

    domains = set()
    ips = set()

    for url in urls:
        parsed = urlparse(url)
        hostname = parsed.hostname

        if not hostname:
            continue

        hostname = hostname.lower()

        try:
            ipaddress = __import__("ipaddress")
            ipaddress.ip_address(hostname)
            ips.add(hostname)
        except Exception:
            domains.add(hostname)

    for ip in IP_RE.findall(text or ""):
        try:
            ipaddress = __import__("ipaddress")
            ipaddress.ip_address(ip)
            ips.add(ip)
        except Exception:
            pass

    for domain in DOMAIN_RE.findall(text or ""):
        domain = domain.lower()

        if domain not in {"example.com"}:
            domains.add(domain)

    return {
        "urls": sorted(urls),
        "domains": sorted(domains),
        "ips": sorted(ips),
    }

In [ ]:
SUSPICIOUS_TLDS = {
    ".zip",
    ".click",
    ".top",
    ".xyz",
    ".buzz",
    ".tk",
    ".ml",
    ".cf",
    ".gq",
}


def analyze_url(url: str) -> dict:

    parsed = urlparse(url)
    host = parsed.hostname or ""

    features = []
    raw_score = 0.0

    if not host:
        return {
            "url": url,
            "score": 0.0,
            "features": ["invalid hostname"]
        }

    if re.fullmatch(r"(?:\d{1,3}\.){3}\d{1,3}", host):
        features.append("IP address used instead of domain")
        raw_score += 0.30

    if host.startswith("xn--") or ".xn--" in host:
        features.append("punycode hostname")
        raw_score += 0.25

    if "@" in url:
        features.append("userinfo/@ character present")
        raw_score += 0.25

    if len(url) > 100:
        features.append("very long URL")
        raw_score += 0.10

    if len(host.split(".")) >= 4:
        features.append("many hostname components")
        raw_score += 0.10

    if host.count("-") >= 3:
        features.append("many hyphens in hostname")
        raw_score += 0.08

    lower_url = url.lower()

    suspicious_keywords = [
        "login",
        "signin",
        "verify",
        "verification",
        "password",
        "secure",
        "account",
        "update",
        "confirm",
    ]

    found_keywords = [
        word for word in suspicious_keywords
        if word in lower_url
    ]

    if found_keywords:
        features.append(
            f"security/account keyword(s): {', '.join(found_keywords)}"
        )
        raw_score += 0.10

    if any(host.endswith(tld) for tld in SUSPICIOUS_TLDS):
        features.append("less-common/suspicious TLD")
        raw_score += 0.10

    score = min(raw_score, 1.0)

    return {
        "url": url,
        "domain": host,
        "score": round(score, 4),
        "features": features,
    }

In [ ]:
def domain_from_address(address: str) -> str:
    _, parsed = parseaddr(address or "")
    if "@" not in parsed:
        return ""

    return parsed.split("@", 1)[1].lower().strip()


def analyze_headers(parsed: dict) -> dict:

    findings = []
    score = 0.0

    sender_domain = domain_from_address(
        parsed.get("from", "")
    )

    reply_domain = domain_from_address(
        parsed.get("reply_to", "")
    )

    return_domain = domain_from_address(
        parsed.get("return_path", "")
    )

    auth_text = (
        parsed.get("authentication_results", "")
        + " "
        + parsed.get("received_spf", "")
    ).lower()

    # SPF
    if "spf=fail" in auth_text:
        findings.append("SPF failed")
        score += 0.20
    elif "spf=softfail" in auth_text:
        findings.append("SPF softfail")
        score += 0.10

    # DKIM
    if "dkim=fail" in auth_text:
        findings.append("DKIM failed")
        score += 0.20

    # DMARC
    if "dmarc=fail" in auth_text:
        findings.append("DMARC failed")
        score += 0.25

    # Reply-To mismatch
    if (
        sender_domain
        and reply_domain
        and sender_domain != reply_domain
    ):
        findings.append(
            "From domain differs from Reply-To domain"
        )
        score += 0.20

    # Return-Path mismatch
    if (
        sender_domain
        and return_domain
        and sender_domain != return_domain
    ):
        findings.append(
            "From domain differs from Return-Path domain"
        )
        score += 0.15

    return {
        "score": round(min(score, 1.0), 4),
        "findings": findings,
        "sender_domain": sender_domain,
        "reply_domain": reply_domain,
        "return_path_domain": return_domain,
    }

In [ ]:
tavily_tool_base = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="basic",
    include_answer=True,
    include_raw_content=False,
    include_usage=True,
)


@tool
def tavily_search(query: str) -> str:
    """
    Search the web for external evidence about a suspicious
    sender, domain, URL, organization impersonation claim,
    or phishing report.

    Use this only for targeted security investigation.
    """

    query = query.strip()

    if not query:
        return json.dumps({
            "error": "Empty search query."
        })

    try:
        result = tavily_tool_base.invoke({
            "query": query
        })

        # Keep only compact evidence.
        compact_results = []

        for item in result.get("results", [])[:5]:
            compact_results.append({
                "title": item.get("title"),
                "url": item.get("url"),
                "content": (item.get("content") or "")[:900],
                "score": item.get("score"),
            })

        output = {
            "query": query,
            "answer": result.get("answer"),
            "results": compact_results,
        }

        return json.dumps(output, ensure_ascii=False)

    except Exception as exc:
        return json.dumps({
            "error": str(exc)
        })

In [ ]:
OTX_BASE_URL = "https://otx.alienvault.com/api/v1"


def detect_indicator_type(indicator: str) -> str:

    indicator = indicator.strip()

    try:
        ipaddress = __import__("ipaddress")
        ipaddress.ip_address(indicator)

        if ":" in indicator:
            return "IPv6"

        return "IPv4"

    except Exception:
        pass

    if indicator.lower().startswith(
        ("http://", "https://")
    ):
        return "url"

    return "domain"


@tool
def otx_lookup(indicator: str) -> str:
    """
    Query AlienVault OTX for threat intelligence about a
    domain, URL, IPv4 address, or IPv6 address.
    """

    indicator = indicator.strip()

    if not indicator:
        return json.dumps({
            "error": "Empty indicator."
        })

    indicator_type = detect_indicator_type(indicator)

    encoded_indicator = requests.utils.quote(
        indicator,
        safe=""
    )

    if indicator_type == "domain":
        endpoint = (
            f"{OTX_BASE_URL}/indicators/domain/"
            f"{encoded_indicator}/general"
        )

    elif indicator_type == "url":
        endpoint = (
            f"{OTX_BASE_URL}/indicators/url/"
            f"{encoded_indicator}/general"
        )

    elif indicator_type == "IPv4":
        endpoint = (
            f"{OTX_BASE_URL}/indicators/IPv4/"
            f"{encoded_indicator}/general"
        )

    else:
        endpoint = (
            f"{OTX_BASE_URL}/indicators/IPv6/"
            f"{encoded_indicator}/general"
        )

    headers = {
        "X-OTX-API-KEY": os.environ["OTX_API_KEY"],
        "Accept": "application/json",
    }

    try:
        response = requests.get(
            endpoint,
            headers=headers,
            timeout=20,
        )

        if response.status_code in (400, 404):
            return json.dumps({
                "indicator": indicator,
                "indicator_type": indicator_type,
                "pulse_count": 0,
                "pulses": [],
                "note": (
                    "no threat intel available "
                    f"(HTTP {response.status_code})"
                ),
            })

        response.raise_for_status()
        data = response.json()

        pulse_info = data.get("pulse_info", {})
        pulses = pulse_info.get("pulses", [])

        compact_pulses = []

        for pulse in pulses[:10]:
            compact_pulses.append({
                "name": pulse.get("name"),
                "id": pulse.get("id"),
                "description": (
                    pulse.get("description") or ""
                )[:500],
                "tags": pulse.get("tags", [])[:10],
            })

        output = {
            "indicator": indicator,
            "indicator_type": indicator_type,
            "pulse_count": pulse_info.get("count", 0),
            "pulses": compact_pulses,
            "sections": data.get("sections", []),
            "reputation": data.get("reputation"),
        }

        return json.dumps(
            output,
            ensure_ascii=False
        )

    except Exception as exc:
        return json.dumps({
            "indicator": indicator,
            "error": str(exc),
        })


In [ ]:
print("Testing Tavily...\n")

print(
    tavily_search.invoke(
        {"query": "example.com phishing"}
    )
[:2000]
)

print("\nTesting OTX...\n")

print(
    otx_lookup.invoke(
        {"indicator": "example.com"}
    )
[:2000]
)

Testing Tavily...

{"query": "example.com phishing", "answer": "The sources do not provide specific information about phishing examples involving example.com. They mention phishing examples from various companies like Monday.com, Home Depot, FedEx, DocuSign, DoorDash, GoDaddy, Apple, PayPal, and others, but none of these examples involve example.com. The sources discuss different types of phishing attacks, including vishing, email phishing, spear phishing, and clone phishing, but none specifically mention example.com phishing.", "results": [{"title": "Phishing Email Examples With Red Flags | Hook Security", "url": "https://www.hooksecurity.co/phishing-email-examples", "content": "monday.com · Social Proofmonday.com phishing email example“Monday.com - Invitation to join Board” from support@shared-document.comHome Depot phishing email example: “Home Depot Super Coupon” from homedepot@thecompliancesolutions.com, offering $5 off with a Get Coupon button. [...] FedEx · UrgencyFedEx phishing

In [ ]:
JEV_URL = "https://openrouter.ai/api/alpha/decisions"


def run_jev(parsed_email: dict, iocs: dict, headers: dict) -> dict:

    api_key = os.environ.get("OPENROUTER_API_KEY")

    if not USE_JEV or not api_key:
        return {
            "available": False,
            "score": None,
            "signals": {},
            "error": "Jev disabled or OpenRouter key unavailable."
        }

    body_text = parsed_email.get("body", "")

    state = {
        "email": {
            "subject": parsed_email.get("subject", ""),
            "sender": parsed_email.get(
                "sender_address", ""
            ),
            "reply_to": parsed_email.get(
                "reply_to_address", ""
            ),
            "body": body_text[:10000],
        },
        "indicators": {
            "urls": iocs.get("urls", [])[:10],
            "domains": iocs.get("domains", [])[:10],
            "ips": iocs.get("ips", [])[:10],
        },
        "header_findings": headers,
    }

    questions = {

        "phishing_score": {
            "type": "score",
            "instructions": (
                "How strongly does this email exhibit "
                "characteristics of a phishing email? "
                "Evaluate the email content, sender identity, "
                "requested action, links and supplied header findings."
            ),
            "criteria": [
                "Very likely legitimate. No meaningful phishing indicators.",
                "Mostly legitimate. Minor suspicious characteristics only.",
                "Ambiguous. Both benign and suspicious characteristics are present.",
                "Likely phishing. Multiple substantive phishing indicators are present.",
                "Strongly phishing. Clear deceptive or credential/theft behavior is present."
            ],
        },

        "urgency": {
            "type": "noul",
            "instructions": (
                "Does the email use urgency, threats, deadlines, "
                "or pressure to make the recipient act?"
            ),
            "criteria": {
                "true": "The email uses pressure, threats, deadlines, or urgency.",
                "false": "The email does not meaningfully use pressure or urgency."
            }
        },

        "credential_request": {
            "type": "noul",
            "instructions": (
                "Does the email ask the recipient to provide, "
                "verify, reset, or submit credentials or sensitive information?"
            ),
            "criteria": {
                "true": "The email requests credentials, authentication data, financial information, or similar sensitive information.",
                "false": "The email does not request such information."
            }
        },

        "impersonation": {
            "type": "noul",
            "instructions": (
                "Does the email appear to impersonate an organization, "
                "brand, service, employee, or trusted party?"
            ),
            "criteria": {
                "true": "The email presents itself as another trusted party or attempts to imitate one.",
                "false": "The sender identity does not appear to imitate another trusted party."
            }
        },

        "suspicious_action": {
            "type": "noul",
            "instructions": (
                "Does the email ask the recipient to click a link, "
                "open something, send information, or otherwise perform "
                "an action that could expose them to harm?"
            ),
            "criteria": {
                "true": "The requested action could plausibly expose the recipient to phishing or another security risk.",
                "false": "The requested action is not materially suspicious."
            }
        },
    }

    payload = {
        "model": "~typesafe/jev-latest",
        "state": state,
        "questions": questions,
    }

    try:

        response = requests.post(
            JEV_URL,
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
            },
            json=payload,
            timeout=45,
        )

        response.raise_for_status()

        data = response.json()
        answers = data["answers"]

        raw_score = answers["phishing_score"]["score"]

        # 0-4 Jev scale → 0-1 phishing score
        normalized_score = float(raw_score) / 4.0

        signals = {}

        for key in [
            "urgency",
            "credential_request",
            "impersonation",
            "suspicious_action",
        ]:
            signals[key] = float(
                answers[key]["noul"]
            )

        return {
            "available": True,
            "score": round(normalized_score, 4),
            "raw_score": raw_score,
            "signals": signals,
            "confidence": answers[
                "phishing_score"
            ].get("confidence"),
            "usage": data.get("usage"),
        }

    except Exception as exc:

        return {
            "available": False,
            "score": None,
            "signals": {},
            "error": str(exc),
        }

In [ ]:
parsed_test = parse_email(TEST_EMAIL)
iocs_test = extract_iocs(TEST_EMAIL)
headers_test = analyze_headers(parsed_test)

jev_test = run_jev(
    parsed_test,
    iocs_test,
    headers_test
)

print(json.dumps(
    jev_test,
    indent=2,
    ensure_ascii=False
))

{
  "available": true,
  "score": 0.925,
  "raw_score": 3.7,
  "signals": {
    "urgency": 0.97,
    "credential_request": 0.81,
    "impersonation": 0.56,
    "suspicious_action": 0.97
  },
  "confidence": 0.75,
  "usage": {
    "input_tokens": 826,
    "output_tokens": 92,
    "cost": 3.4692e-05
  }
}


In [ ]:
class PhishingState(TypedDict, total=False):

    # Raw input
    email_text: str

    # Parsed email
    parsed_email: dict

    # IOCs
    iocs: dict
    url_findings: list

    # Deterministic ML
    ml_score: float

    # Header analysis
    header_findings: dict

    # Jev
    jev: dict

    # Agent messages
    messages: Annotated[
        list,
        add_messages
    ]

    # Investigation
    investigation_round: int
    tavily_results: list
    otx_results: list

    # Gemini investigation
    investigation_score: float
    investigation_reasons: list

    # Evidence
    evidence_nodes: list
    evidence_edges: list

    # Final
    combined_score: float
    verdict: str
    confidence: float
    final_reasons: list
    final_explanation: str
    recommendation: str
    explanation_source: str


In [ ]:
def parse_email_node(state: PhishingState):

    parsed = parse_email(
        state["email_text"]
    )

    return {
        "parsed_email": parsed
    }


def extract_iocs_node(state: PhishingState):

    parsed = state["parsed_email"]

    text = (
        parsed.get("subject", "")
        + "\n"
        + parsed.get("body", "")
    )

    # Include headers as IOC-bearing text too.
    text += "\n" + str(parsed.get("headers", {}))

    iocs = extract_iocs(text)

    return {
        "iocs": iocs
    }


def url_analysis_node(state: PhishingState):

    findings = [
        analyze_url(url)
        for url in state["iocs"].get("urls", [])
    ]

    return {
        "url_findings": findings
    }


def header_analysis_node(state: PhishingState):

    return {
        "header_findings": analyze_headers(
            state["parsed_email"]
        )
    }


def ml_predict_node(state: PhishingState):

    score = phishing_model.predict_proba(
        [state["email_text"]]
    )[0][1]

    return {
        "ml_score": round(
            float(score),
            4
        )
    }


def jev_node(state: PhishingState):

    result = run_jev(
        state["parsed_email"],
        state["iocs"],
        state["header_findings"],
    )

    return {
        "jev": result
    }

In [ ]:
PRIMARY_BRAIN_MODEL = "gemini-3.8-flash"
FALLBACK_BRAIN_MODEL = "gemini-3.1-flash-lite"
SCORER_MODEL = "gemini-3.1-flash-lite"

gemini_brain = ChatGoogleGenerativeAI(
    model=PRIMARY_BRAIN_MODEL,
    google_api_key=GEMINI_API_KEY,
    temperature=0,
    thinking_level="low",
)

gemini_brain_fallback = ChatGoogleGenerativeAI(
    model=FALLBACK_BRAIN_MODEL,
    google_api_key=GEMINI_API_KEY,
    temperature=0,
    thinking_level="minimal",
)

gemini_lite = ChatGoogleGenerativeAI(
    model=SCORER_MODEL,
    google_api_key=GEMINI_API_KEY,
    temperature=0,
    thinking_level="minimal",
)

investigation_tools = [
    tavily_search,
    otx_lookup,
]

gemini_brain_with_tools = gemini_brain.bind_tools(
    investigation_tools
)

gemini_brain_fallback_with_tools = (
    gemini_brain_fallback.bind_tools(
        investigation_tools
    )
)


def is_rate_limit_error(exc: Exception) -> bool:
    text = str(exc).lower()
    return (
        "429" in text
        or "resource_exhausted" in text
    )


def invoke_brain(messages):
    try:
        return gemini_brain_with_tools.invoke(
            messages
        )

    except Exception as exc:
        if not is_rate_limit_error(exc):
            raise

        print(
            f"{PRIMARY_BRAIN_MODEL} rate limited, "
            f"switching brain to "
            f"{FALLBACK_BRAIN_MODEL}."
        )

        try:
            return (
                gemini_brain_fallback_with_tools.invoke(
                    messages
                )
            )

        except Exception as fallback_exc:
            if not is_rate_limit_error(fallback_exc):
                raise

            print(
                f"{FALLBACK_BRAIN_MODEL} also rate "
                "limited. Skipping agentic "
                "investigation."
            )

            return AIMessage(
                content=(
                    "Investigation skipped: both brain "
                    "models are rate limited. Scoring "
                    "will proceed with deterministic "
                    "evidence only."
                )
            )


INVESTIGATOR_SYSTEM = """
You are the threat-investigation agent in a phishing-email
analysis system.

Your task is to investigate suspicious indicators using the
available tools.

Available tools:
1. tavily_search
2. otx_lookup

Rules:

- You have at most two investigation turns.
- Make multiple independent tool calls in a single turn
  whenever possible.
- OTX threat-intel results for the extracted indicators
  are already attached to your evidence. Do not re-query
  OTX for indicators that already have results.
- Do not invent external evidence.
- Use Tavily for targeted web investigation.
- Use OTX for domains, URLs, and IP indicators.
- Prefer investigating extracted domains and URLs over broad searches.
- Search queries should be specific and security-oriented.
- Do not repeatedly investigate the same indicator.
- Stop investigating when there is enough evidence.
- If there is no meaningful IOC, use Tavily only when web evidence
  can materially improve the assessment.
- Do not produce the final phishing verdict yourself.
- Gather evidence for the downstream scoring node.
"""


In [ ]:
# ============================================================
# INVESTIGATOR NODE + ROUTING
# ============================================================

MAX_AGENT_ROUNDS = 2


def investigator_node(state: PhishingState):

    round_number = state.get(
        "investigation_round",
        0
    )

    # --------------------------------------------------------
    # FIRST LLM CALL
    # --------------------------------------------------------
    # We must store the original HumanMessage in state.
    # This is critical for Gemini's tool-call message ordering.
    # --------------------------------------------------------

    if round_number == 0:

        parsed = state["parsed_email"]

        investigation_context = {
            "subject": parsed.get(
                "subject", ""
            ),
            "sender": parsed.get(
                "sender_address", ""
            ),
            "reply_to": parsed.get(
                "reply_to_address", ""
            ),
            "body": parsed.get(
                "body", ""
            )[:12000],
            "urls": state["iocs"].get(
                "urls", []
            ),
            "domains": state["iocs"].get(
                "domains", []
            ),
            "ips": state["iocs"].get(
                "ips", []
            ),
            "url_findings": state.get(
                "url_findings", []
            ),
            "header_findings": state.get(
                "header_findings", {}
            ),
            "ml_score": state.get(
                "ml_score"
            ),
            "jev": state.get(
                "jev", {}
            ),
            "otx_prefetch_results": state.get(
                "otx_results", []
            ),
        }

        system_message = SystemMessage(
            content=INVESTIGATOR_SYSTEM
        )

        human_message = HumanMessage(
            content=(
                "Investigate this email.\n\n"
                "Initial structured evidence:\n"
                + json.dumps(
                    investigation_context,
                    indent=2,
                    ensure_ascii=False,
                )
            )
        )

        messages_for_llm = [
            system_message,
            human_message,
        ]

        response = invoke_brain(
            messages_for_llm
        )

        # IMPORTANT:
        # Store the system + human + AI messages.
        # ToolNode will append the ToolMessage afterward.
        return {
            "messages": [
                system_message,
                human_message,
                response,
            ],
            "investigation_round": 1,
        }

    # --------------------------------------------------------
    # SUBSEQUENT CALLS
    # --------------------------------------------------------
    # state["messages"] already contains:
    #
    # System
    # Human
    # AI(tool call)
    # Tool(result)
    #
    # Send the history exactly as-is.
    # DO NOT prepend another SystemMessage.
    # --------------------------------------------------------

    messages_for_llm = state["messages"]

    response = invoke_brain(
        messages_for_llm
    )

    return {
        "messages": [response],
        "investigation_round": (
            round_number + 1
        ),
    }


def route_after_investigator(
    state: PhishingState
):

    last_message = state[
        "messages"
    ][-1]

    has_tool_calls = (
        isinstance(
            last_message,
            AIMessage
        )
        and bool(
            last_message.tool_calls
        )
    )

    # Tool execution is allowed only while the
    # MAX_AGENT_ROUNDS brain-call budget is unspent.
    # Tool calls requested on the final round are not
    # executed; the graph moves to evidence scoring.

    if (
        has_tool_calls
        and state.get(
            "investigation_round",
            0
        ) < MAX_AGENT_ROUNDS
    ):
        return "tools"

    return "investigation_score"


In [ ]:
# ============================================================
# COLLECT TOOL RESULTS
# ============================================================

def collect_tool_results(
    state: PhishingState
):

    tavily_results = list(
        state.get(
            "tavily_results",
            []
        )
    )

    otx_results = list(
        state.get(
            "otx_results",
            []
        )
    )

    for message in state.get(
        "messages",
        []
    ):

        if not isinstance(
            message,
            ToolMessage
        ):
            continue

        try:
            data = json.loads(
                message.content
            )
        except Exception:
            continue

        tool_name = (
            message.name or ""
        ).lower()

        if "tavily" in tool_name:

            if data not in tavily_results:
                tavily_results.append(data)

        elif "otx" in tool_name:

            if data not in otx_results:
                otx_results.append(data)

    return {
        "tavily_results": tavily_results,
        "otx_results": otx_results,
    }

In [ ]:
# ============================================================
# DETERMINISTIC IOC PREFETCH
# ============================================================

from concurrent.futures import ThreadPoolExecutor

PREFETCH_OTX_LIMIT = 8


def _otx_fetch(indicator: str) -> dict:

    try:
        return json.loads(
            otx_lookup.invoke(
                {"indicator": indicator}
            )
        )

    except Exception as exc:
        return {
            "indicator": indicator,
            "error": str(exc),
        }


def prefetch_iocs_node(state: PhishingState):

    iocs = state.get("iocs", {})

    indicators = (
        list(iocs.get("domains", []))
        + list(iocs.get("urls", []))
        + list(iocs.get("ips", []))
    )

    seen = set()
    unique = []

    for indicator in indicators:
        if indicator not in seen:
            seen.add(indicator)
            unique.append(indicator)

    indicators = unique[:PREFETCH_OTX_LIMIT]

    if not indicators:
        return {"otx_results": []}

    with ThreadPoolExecutor(max_workers=5) as pool:
        results = list(
            pool.map(_otx_fetch, indicators)
        )

    return {"otx_results": results}


In [ ]:
class InvestigationAssessment(BaseModel):

    phishing_score: float = Field(
        ge=0.0,
        le=1.0,
        description=(
            "Probability-like phishing risk score from 0 to 1. "
            "This is an assessment score, not a calibrated probability."
        )
    )

    evidence_quality: str = Field(
        description=(
            "Overall quality of external investigation evidence: "
            "low, medium, or high."
        )
    )

    reasons: list[str] = Field(
        description=(
            "Specific evidence-backed reasons supporting the score."
        )
    )

In [ ]:
gemini_scorer = gemini_lite.with_structured_output(
    InvestigationAssessment
)


def investigation_score_node(
    state: PhishingState
):

    parsed = state["parsed_email"]

    evidence_packet = {
        "email": {
            "subject": parsed.get(
                "subject", ""
            ),
            "sender": parsed.get(
                "sender_address", ""
            ),
            "reply_to": parsed.get(
                "reply_to_address", ""
            ),
            "body": parsed.get(
                "body", ""
            )[:12000],
        },

        "url_findings": state.get(
            "url_findings",
            []
        ),

        "header_findings": state.get(
            "header_findings",
            {}
        ),

        "ml_score": state.get(
            "ml_score"
        ),

        "jev": state.get(
            "jev",
            {}
        ),

        "tavily_results": state.get(
            "tavily_results",
            []
        ),

        "otx_results": state.get(
            "otx_results",
            []
        ),
    }

    prompt = """
You are the evidence-scoring stage of a phishing-email
investigation.

Assess phishing risk from 0 to 1.

Important:

- 0 means strongly legitimate.
- 1 means strongly phishing.
- Treat local deterministic analysis as factual evidence.
- Treat OTX and web-search findings as external evidence.
- Do not invent evidence.
- Do not blindly trust a single source.
- Consider evidence quality and contradictions.
- The score must reflect the evidence available to you.
- This score will later be mathematically fused with independent
  ML and Jev scores.
"""

    try:
        result = gemini_scorer.invoke([
            SystemMessage(
                content=prompt
            ),
            HumanMessage(
                content=json.dumps(
                    evidence_packet,
                    indent=2,
                    ensure_ascii=False,
                )
            )
        ])

    except Exception as exc:
        print(f"Evidence scorer failed: {exc}")

        return {
            "investigation_score": None,
            "investigation_reasons": [
                "Evidence scoring unavailable. "
                "Fusion will reweight the remaining "
                "subsystem scores."
            ],
        }

    return {
        "investigation_score": round(
            result.phishing_score,
            4
        ),
        "investigation_reasons": result.reasons,
    }


In [ ]:
def build_evidence_graph(
    state: PhishingState
):

    nodes = []
    edges = []

    def add_node(
        node_id,
        node_type,
        label,
        metadata=None
    ):
        nodes.append({
            "id": node_id,
            "type": node_type,
            "label": label,
            "metadata": metadata or {},
        })

    def add_edge(
        source,
        target,
        relation
    ):
        edges.append({
            "source": source,
            "target": target,
            "relation": relation,
        })

    # Root email
    add_node(
        "email",
        "email",
        "EMAIL",
        {
            "subject": state[
                "parsed_email"
            ].get("subject", "")
        }
    )

    # Sender
    sender = state[
        "parsed_email"
    ].get(
        "sender_address",
        ""
    )

    if sender:
        add_node(
            "sender",
            "sender",
            sender
        )

        add_edge(
            "email",
            "sender",
            "sent_by"
        )

    # URLs
    for i, url_info in enumerate(
        state.get(
            "url_findings",
            []
        )
    ):

        url_id = f"url_{i}"

        add_node(
            url_id,
            "url",
            url_info["url"],
            url_info
        )

        add_edge(
            "email",
            url_id,
            "contains"
        )

        domain = url_info.get(
            "domain"
        )

        if domain:
            domain_id = f"domain_{i}"

            add_node(
                domain_id,
                "domain",
                domain
            )

            add_edge(
                url_id,
                domain_id,
                "belongs_to"
            )

    # Header evidence
    for i, finding in enumerate(
        state.get(
            "header_findings",
            {}
        ).get(
            "findings",
            []
        )
    ):

        header_id = (
            f"header_{i}"
        )

        add_node(
            header_id,
            "header",
            finding
        )

        add_edge(
            "email",
            header_id,
            "indicates"
        )

    # OTX
    for i, result in enumerate(
        state.get(
            "otx_results",
            []
        )
    ):

        if result.get("error"):
            continue

        node_id = f"otx_{i}"

        indicator = result.get(
            "indicator"
        )

        pulse_count = result.get(
            "pulse_count",
            0
        )

        add_node(
            node_id,
            "threat_intel",
            (
                f"OTX: {indicator}"
                f" ({pulse_count} pulses)"
            ),
            result
        )

        matching_domain = None

        for domain_idx, url_info in enumerate(
            state.get(
                "url_findings",
                []
            )
        ):
            if (
                url_info.get("domain")
                == indicator
            ):
                matching_domain = (
                    f"domain_{domain_idx}"
                )
                break

        if matching_domain:
            add_edge(
                matching_domain,
                node_id,
                "investigated_by"
            )

        else:
            add_edge(
                "email",
                node_id,
                "investigated_indicator"
            )

    # Tavily
    for i, result in enumerate(
        state.get(
            "tavily_results",
            []
        )
    ):

        if result.get("error"):
            continue

        query = result.get(
            "query",
            "search"
        )

        node_id = f"search_{i}"

        add_node(
            node_id,
            "web_result",
            f"Tavily: {query}",
            {
                "query": query,
                "answer": result.get("answer"),
                "result_count": len(
                    result.get(
                        "results",
                        []
                    )
                ),
                "results": result.get(
                    "results",
                    []
                ),
            }
        )

        add_edge(
            "email",
            node_id,
            "external_evidence"
        )

    # Model scores
    add_node(
        "ml_model",
        "model",
        f"ML: {state.get('ml_score', 0):.2f}"
    )

    add_node(
        "jev_model",
        "model",
        f"Jev: {state.get('jev', {}).get('score') if state.get('jev', {}).get('score') is not None else 'N/A'}"
    )

    add_node(
        "gemini_model",
        "model",
        f"Gemini: {state.get('investigation_score') or 0:.2f}"
    )

    add_edge(
        "email",
        "ml_model",
        "classified_by"
    )

    add_edge(
        "email",
        "jev_model",
        "judged_by"
    )

    add_edge(
        "email",
        "gemini_model",
        "investigated_by"
    )

    # Final
    add_node(
        "final",
        "verdict",
        state.get(
            "verdict",
            "UNKNOWN"
        ),
        {
            "combined_score":
                state.get(
                    "combined_score"
                ),
            "confidence":
                state.get(
                    "confidence"
                ),
        }
    )

    add_edge(
        "ml_model",
        "final",
        "supports"
    )

    add_edge(
        "jev_model",
        "final",
        "supports"
    )

    add_edge(
        "gemini_model",
        "final",
        "supports"
    )

    return {
        "evidence_nodes": nodes,
        "evidence_edges": edges,
    }


In [ ]:
ML_WEIGHT = 0.45
JEV_WEIGHT = 0.25
GEMINI_WEIGHT = 0.30

PHISHING_THRESHOLD = 0.70


def combine_scores_node(
    state: PhishingState
):

    components = []

    ml_score = state.get(
        "ml_score"
    )

    if ml_score is not None:
        components.append(
            (
                ML_WEIGHT,
                float(ml_score)
            )
        )

    jev_score = state.get(
        "jev",
        {}
    ).get("score")

    if jev_score is not None:
        components.append(
            (
                JEV_WEIGHT,
                float(jev_score)
            )
        )

    gemini_score = state.get(
        "investigation_score"
    )

    if gemini_score is not None:
        components.append(
            (
                GEMINI_WEIGHT,
                float(gemini_score)
            )
        )

    if not components:
        raise ValueError(
            "No scoring subsystem available."
        )

    total_weight = sum(
        weight
        for weight, _ in components
    )

    combined_score = (
        sum(
            weight * score
            for weight, score in components
        )
        / total_weight
    )

    scores = [
        score
        for _, score in components
    ]

    # Consensus agreement:
    # 1 = all systems agree
    # 0 = maximum disagreement
    if len(scores) > 1:
        std = statistics.pstdev(scores)
        agreement = 1.0 - min(
            std / 0.5,
            1.0
        )
    else:
        agreement = 0.5

    # Distance from neutral.
    margin = min(
        abs(combined_score - 0.5) * 2,
        1.0
    )

    confidence = (
        0.5 * agreement
        + 0.5 * margin
    )

    verdict = (
        "PHISHING"
        if combined_score >= PHISHING_THRESHOLD
        else "LIKELY LEGITIMATE"
    )

    return {
        "combined_score": round(
            combined_score,
            4
        ),
        "verdict": verdict,
        "confidence": round(
            confidence,
            4
        ),
    }

In [ ]:
GEMINI_EXPLANATION = True


class FinalExplanation(BaseModel):

    summary: str

    reasons: list[str]

    recommendation: str


final_explainer = gemini_lite.with_structured_output(
    FinalExplanation
)


def fallback_explanation(state: PhishingState) -> dict:

    reasons = list(
        state.get(
            "investigation_reasons",
            []
        )
    ) + list(
        state.get(
            "header_findings",
            {}
        ).get(
            "findings",
            []
        )
    )

    summary = (
        f"Deterministic assessment: {state['verdict']} "
        f"at combined score {state['combined_score']:.3f} "
        f"with confidence {state['confidence']:.3f}."
    )

    recommendation = (
        "Do not click links, open attachments, or submit "
        "credentials from this email. Report it to your "
        "security team and delete it."
    )

    return {
        "final_explanation": summary,
        "final_reasons": reasons,
        "recommendation": recommendation,
        "explanation_source": "python",
    }


def final_explanation_node(
    state: PhishingState
):

    if not GEMINI_EXPLANATION:
        return fallback_explanation(state)

    packet = {
        "verdict": state["verdict"],
        "combined_score": state[
            "combined_score"
        ],
        "confidence": state[
            "confidence"
        ],

        "ml_score": state.get(
            "ml_score"
        ),

        "jev": state.get(
            "jev",
            {}
        ),

        "gemini_investigation_score":
            state.get(
                "investigation_score"
            ),

        "header_findings":
            state.get(
                "header_findings",
                {}
            ),

        "url_findings":
            state.get(
                "url_findings",
                []
            ),

        "investigation_reasons":
            state.get(
                "investigation_reasons",
                []
            ),

        "otx_results":
            state.get(
                "otx_results",
                []
            ),

        "tavily_results":
            state.get(
                "tavily_results",
                []
            ),
    }

    prompt = """
You are the final reporting layer of an explainable
phishing-email detection system.

Generate a concise security report.

Important:
- Never invent evidence.
- Only use the supplied evidence.
- Do not modify the supplied verdict, combined score,
  or confidence.
- Distinguish deterministic findings from model judgments.
- Explain why the systems reached their result.
- Give practical defensive advice.

The report should be understandable to a technically
competent human reviewer.
"""

    try:
        result = final_explainer.invoke([
            SystemMessage(
                content=prompt
            ),
            HumanMessage(
                content=json.dumps(
                    packet,
                    indent=2,
                    ensure_ascii=False,
                )
            )
        ])

    except Exception as exc:
        print(f"Final explanation failed: {exc}")
        return fallback_explanation(state)

    return {
        "final_explanation":
            result.summary,

        "final_reasons":
            result.reasons,

        "recommendation":
            result.recommendation,

        "explanation_source": "gemini",
    }


In [ ]:
tools = [
    tavily_search,
    otx_lookup,
]

tool_node = ToolNode(
    tools,
    handle_tool_errors=True,
)


builder = StateGraph(
    PhishingState
)

# Deterministic preprocessing
builder.add_node(
    "parse_email",
    parse_email_node
)

builder.add_node(
    "extract_iocs",
    extract_iocs_node
)

builder.add_node(
    "url_analysis",
    url_analysis_node
)

builder.add_node(
    "ml_predict",
    ml_predict_node
)

builder.add_node(
    "header_analysis",
    header_analysis_node
)

builder.add_node(
    "jev",
    jev_node
)

builder.add_node(
    "ioc_prefetch",
    prefetch_iocs_node
)

# Agentic investigation
builder.add_node(
    "investigator",
    investigator_node
)

builder.add_node(
    "tools",
    tool_node
)

builder.add_node(
    "collect_tool_results",
    collect_tool_results
)

builder.add_node(
    "investigation_score",
    investigation_score_node
)

# Output
builder.add_node(
    "build_evidence_graph",
    build_evidence_graph
)

builder.add_node(
    "combine_scores",
    combine_scores_node
)

builder.add_node(
    "final_explanation",
    final_explanation_node
)


# Flow
builder.add_edge(
    START,
    "parse_email"
)

builder.add_edge(
    "parse_email",
    "extract_iocs"
)

builder.add_edge(
    "extract_iocs",
    "url_analysis"
)

builder.add_edge(
    "url_analysis",
    "ml_predict"
)

builder.add_edge(
    "ml_predict",
    "header_analysis"
)

builder.add_edge(
    "header_analysis",
    "jev"
)

builder.add_edge(
    "jev",
    "ioc_prefetch"
)

builder.add_edge(
    "ioc_prefetch",
    "investigator"
)

# Investigator ↔ tools loop
builder.add_conditional_edges(
    "investigator",
    route_after_investigator,
    {
        "tools": "tools",
        "investigation_score":
            "investigation_score",
    }
)

builder.add_edge(
    "tools",
    "collect_tool_results"
)

builder.add_edge(
    "collect_tool_results",
    "investigator"
)

# Final pipeline
builder.add_edge(
    "investigation_score",
    "build_evidence_graph"
)

builder.add_edge(
    "build_evidence_graph",
    "combine_scores"
)

builder.add_edge(
    "combine_scores",
    "final_explanation"
)

builder.add_edge(
    "final_explanation",
    END
)

graph = builder.compile()

print("LangGraph compiled successfully.")


In [ ]:
try:
    display(
        graph.get_graph().draw_mermaid()
    )
except Exception as e:
    print("Graph visualization unavailable:", e)

'---\nconfig:\n  flowchart:\n    curve: linear\n---\ngraph TD;\n\t__start__([<p>__start__</p>]):::first\n\tparse_email(parse_email)\n\textract_iocs(extract_iocs)\n\turl_analysis(url_analysis)\n\tml_predict(ml_predict)\n\theader_analysis(header_analysis)\n\tjev(jev)\n\tinvestigator(investigator)\n\ttools(tools)\n\tcollect_tool_results(collect_tool_results)\n\tinvestigation_score(investigation_score)\n\tbuild_evidence_graph(build_evidence_graph)\n\tcombine_scores(combine_scores)\n\tfinal_explanation(final_explanation)\n\t__end__([<p>__end__</p>]):::last\n\t__start__ --> parse_email;\n\tbuild_evidence_graph --> combine_scores;\n\tcollect_tool_results --> investigator;\n\tcombine_scores --> final_explanation;\n\textract_iocs --> url_analysis;\n\theader_analysis --> jev;\n\tinvestigation_score --> build_evidence_graph;\n\tinvestigator -.-> investigation_score;\n\tinvestigator -.-> tools;\n\tjev --> investigator;\n\tml_predict --> header_analysis;\n\tparse_email --> extract_iocs;\n\ttools --

In [ ]:
PHISHING_EMAIL = """
From: Microsoft Account Security <security@micros0ft-support.example>
Reply-To: verification@secure-account-login.example
Return-Path: bounce@secure-account-login.example
Authentication-Results: spf=fail; dkim=fail; dmarc=fail
Subject: URGENT: Your Microsoft account will be suspended

Dear Customer,

We detected unusual activity on your Microsoft account.

Your account will be permanently suspended within 24 hours
unless you verify your identity.

Please confirm your account immediately:

https://micros0ft-support.example/account/verify

Failure to complete verification may result in permanent
loss of access.

Microsoft Security Team
"""

In [ ]:
initial_state = {
    "email_text": PHISHING_EMAIL,
    "messages": [],
    "investigation_round": 0,
}

result = graph.invoke(
    initial_state
)

print("Graph execution completed.")

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 8h46m38.73869762s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier

In [ ]:
ml_score = result.get(
    "ml_score"
)

jev_score = result.get(
    "jev",
    {}
).get(
    "score"
)

gemini_score = result.get(
    "investigation_score"
)

combined_score = result.get(
    "combined_score"
)

confidence = result.get(
    "confidence"
)

print("=" * 60)
print("PHISHING DETECTION SCORES")
print("=" * 60)

if ml_score is not None:
    print(
        f"ML detector:              "
        f"{ml_score:.4f}"
    )
else:
    print("ML detector:              N/A")

if jev_score is not None:
    print(
        f"Jev:                      "
        f"{jev_score:.4f}"
    )
else:
    print("Jev:                      N/A")

if gemini_score is not None:
    print(
        f"Gemini investigation:     "
        f"{gemini_score:.4f}"
    )
else:
    print("Gemini investigation:     N/A")

print(
    f"Combined score:           "
    f"{combined_score:.4f}"
)

print(
    f"Confidence:               "
    f"{confidence:.4f}"
)

print(
    f"Threshold:                "
    f"{PHISHING_THRESHOLD:.2f}"
)

print(
    f"Verdict:                  "
    f"{result['verdict']}"
)


In [ ]:
print("\n" + "=" * 60)
print("REASONS BREAKDOWN")
print("=" * 60)

print("\nHEADER FINDINGS")
for finding in result.get(
    "header_findings",
    {}
).get(
    "findings",
    []
):
    print(f"  • {finding}")


print("\nURL FINDINGS")
for item in result.get(
    "url_findings",
    []
):
    print(
        f"  • {item['url']}"
    )

    for feature in item.get(
        "features",
        []
    ):
        print(
            f"      - {feature}"
        )


print("\nJEV SIGNALS")

for key, value in result.get(
    "jev",
    {}
).get(
    "signals",
    {}
).items():

    print(
        f"  • {key}: {value:.3f}"
    )


print("\nGEMINI INVESTIGATION REASONS")

for reason in result.get(
    "investigation_reasons",
    []
):
    print(
        f"  • {reason}"
    )


print("\nFINAL REPORT REASONS")

for reason in result.get(
    "final_reasons",
    []
):
    print(
        f"  • {reason}"
    )

In [ ]:
print("\u2554" + "\u2550" * 58 + "\u2557")
print("\u2551" + " PHISHING INVESTIGATION REPORT".center(58) + "\u2551")
print("\u255a" + "\u2550" * 58 + "\u255d")

print(
    f"\nVerdict:       {result['verdict']}"
)

print(
    f"Risk score:    {result['combined_score']:.3f}"
)

print(
    f"Confidence:    {result['confidence']:.3f}"
)

print(
    f"Threshold:     {PHISHING_THRESHOLD:.3f}"
)

print("\nSCORE BREAKDOWN")

if ml_score is not None:
    print(
        f"  ML detector             "
        f"{ml_score:.3f}"
    )
else:
    print("  ML detector             N/A")

if jev_score is not None:
    print(
        f"  Jev                     "
        f"{jev_score:.3f}"
    )
else:
    print("  Jev                     N/A")

if gemini_score is not None:
    print(
        f"  Gemini + investigation  "
        f"{gemini_score:.3f}"
    )
else:
    print("  Gemini + investigation  N/A")

print("\nSUMMARY")
print(
    result["final_explanation"]
)

print("\nREASONS")

for reason in result.get(
    "final_reasons",
    []
):
    print(
        f"  \u2022 {reason}"
    )

print("\nRECOMMENDATION")
print(
    result["recommendation"]
)

print(
    f"\nExplanation source: "
    f"{result.get('explanation_source', 'python')}"
)


In [ ]:
def render_evidence_graph(
    result,
    filename=None,
):
    from pyvis.network import Network

    TYPE_COLORS = {
        "email": "#37474f",
        "sender": "#78909c",
        "url": "#8e24aa",
        "domain": "#ba68c8",
        "header": "#e53935",
        "threat_intel": "#fb8c00",
        "web_result": "#43a047",
        "model": "#1e88e5",
        "verdict": "#c62828",
    }

    def short(label, limit=44):
        label = str(label)
        if len(label) <= limit:
            return label
        return label[: limit - 1] + "..."

    verdict = result.get(
        "verdict",
        ""
    )

    net = Network(
        height="900px",
        width="100%",
        directed=True,
        notebook=True,
        cdn_resources="in_line",
    )

    used_types = set()

    for node in result.get(
        "evidence_nodes",
        []
    ):
        node_type = node.get(
            "type",
            "email"
        )
        used_types.add(node_type)

        color = TYPE_COLORS.get(
            node_type,
            "#616161"
        )

        if (
            node_type == "verdict"
            and "LEGITIMATE" in str(verdict).upper()
        ):
            color = "#2e7d32"

        size = 16

        if node_type == "email":
            size = 28
        elif node_type == "model":
            size = 22
        elif node_type == "verdict":
            size = 34

        tooltip = json.dumps(
            node.get("metadata", {}),
            indent=2,
            ensure_ascii=False,
            default=str,
        )

        net.add_node(
            node["id"],
            label=short(
                node.get(
                    "label",
                    node["id"]
                )
            ),
            title=tooltip,
            color=color,
            size=size,
            shape="box",
        )

    for edge in result.get(
        "evidence_edges",
        []
    ):
        net.add_edge(
            edge["source"],
            edge["target"],
            label=edge["relation"],
            title=edge["relation"],
        )

    net.set_options("""
    {
      "layout": {
        "hierarchical": {
          "enabled": true,
          "direction": "LR",
          "sortMethod": "directed",
          "levelSeparation": 170,
          "nodeSpacing": 100,
          "treeSpacing": 150
        }
      },
      "nodes": {
        "font": {
          "size": 13
        },
        "borderWidth": 1
      },
      "edges": {
        "arrows": {
          "to": {
            "enabled": true
          }
        },
        "font": {
          "size": 10
        },
        "smooth": {
          "enabled": false
        }
      },
      "physics": {
        "enabled": false
      },
      "interaction": {
        "tooltipDelay": 100
      }
    }
    """)

    html = net.generate_html()

    legend_rows = "".join(
        "<span style='display:inline-block;"
        "width:10px;height:10px;"
        f"background:{TYPE_COLORS[t]};"
        "margin-right:4px;'></span>"
        f"{t}&nbsp;&nbsp;&nbsp;"
        for t in sorted(used_types)
    )

    legend = (
        "<div style='position:fixed;top:8px;"
        "left:8px;background:#ffffffdd;"
        "padding:8px 12px;border:1px solid #bbb;"
        f"font:13px sans-serif;z-index:9999;'>"
        f"{legend_rows}</div>"
    )

    html = html.replace(
        "</body>",
        legend + "</body>",
        1
    )

    if filename:
        Path(filename).write_text(
            html,
            encoding="utf-8"
        )
        print(f"Saved {filename}")

        return html

    display(HTML(html))

    return html


In [ ]:
render_evidence_graph(result)

In [ ]:
print("=" * 60)
print("AGENT TOOL TRACE")
print("=" * 60)

for i, message in enumerate(
    result.get(
        "messages",
        []
    )
):

    if isinstance(
        message,
        HumanMessage
    ):
        print(
            f"\n[{i}] USER INPUT"
        )

    elif isinstance(
        message,
        AIMessage
    ):

        if message.tool_calls:

            print(
                f"\n[{i}] GEMINI TOOL CALL"
            )

            for call in message.tool_calls:
                print(
                    f"  → {call['name']}"
                )
                print(
                    f"    args: {call['args']}"
                )

        else:
            print(
                f"\n[{i}] GEMINI"
            )

            content = str(
                message.content
            )

            print(
                content[:1500]
            )

    elif isinstance(
        message,
        ToolMessage
    ):

        print(
            f"\n[{i}] TOOL RESULT: "
            f"{message.name}"
        )

        print(
            str(message.content)[:1000]
        )

In [ ]:
def analyze_email(
    email_text: str
) -> dict:

    result = graph.invoke({
        "email_text": email_text,
        "messages": [],
        "investigation_round": 0,
    })

    return result